# RAGAS Evaluation

In [1]:
import pandas as pd
from datasets import Dataset
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
)

In [2]:
!pwd

/home/baptvit/Documents/mestrado/master-experiments/evaluations


In [4]:
df = pd.read_csv(
    "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/silver/Jacklyn830_Veum823_e0e1f21a-22a7-d166-7bb1-63f6bbce1a32.csv"
)

In [5]:
df.columns

Index(['Unnamed: 0', 'experiment_id', 'timestamp_output_step', 'full_response',
       'system_promt', 'input', 'output', 'timestamp_search_step', 'latency_s',
       'query', 'similarity_threshold', 'k', 'embedding_model',
       'timestamp_token_step', 'total_cost', 'total_tokens',
       'successful_requests', 'completion_tokens', 'prompt_tokens',
       'total_time', 'timestamp_tool_step', 'records', 'caracteres_count',
       'pass_map_reduce', 'resource_type', 'main_keys', 'user_query',
       'llm_token_limit', 'strategy_name'],
      dtype='object')

In [11]:
# df["full_input"] = "System prompt:: " + df["system_promt"] + "\n User Question: " + df["input"]

In [12]:
df["full_input"] = df["input"]

In [13]:
eval_df = df[["experiment_id", "strategy_name", "full_input", "output", "records"]]

In [14]:
eval_df["user_input"] = eval_df["full_input"]
eval_df["question"] = eval_df["full_input"]

eval_df["response"] = eval_df["output"]
eval_df["contexts"] = eval_df.records.apply(lambda x: x[0:-1].split("\n"))

eval_df["reference_contexts"] = eval_df.records.apply(lambda x: x[0:-1].split("\n"))
eval_df["retrieved_contexts"] = eval_df.records.apply(lambda x: x[0:-1].split("\n"))

/tmp/ipykernel_22499/4287056615.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df["user_input"] = eval_df["full_input"]
/tmp/ipykernel_22499/4287056615.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df["question"] = eval_df["full_input"]
/tmp/ipykernel_22499/4287056615.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pan

In [15]:
eval_df

,experiment_id,strategy_name,full_input,output,records,user_input,question,response,contexts,reference_contexts,retrieved_contexts
0,Q6:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you provide a breakdown of my medical bills ?,This is a summary of your medical bills:\n\n**...,"{'Main health record': 'Resource Type: Claim,\...",Can you provide a breakdown of my medical bills ?,Can you provide a breakdown of my medical bills ?,This is a summary of your medical bills:\n\n**...,"[{'Main health record': 'Resource Type: Claim,...","[{'Main health record': 'Resource Type: Claim,...","[{'Main health record': 'Resource Type: Claim,..."
1,Q8:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you summarize my immunization history ?,Your immunization history shows that you recei...,{'Main health record': 'Resource Type: Immuniz...,Can you summarize my immunization history ?,Can you summarize my immunization history ?,Your immunization history shows that you recei...,[{'Main health record': 'Resource Type: Immuni...,[{'Main health record': 'Resource Type: Immuni...,[{'Main health record': 'Resource Type: Immuni...
2,Q7:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,"What procedures have I undergone recently, and...","You have undergone several procedures, here is...",{'Main health record': 'Resource Type: Procedu...,"What procedures have I undergone recently, and...","What procedures have I undergone recently, and...","You have undergone several procedures, here is...",[{'Main health record': 'Resource Type: Proced...,[{'Main health record': 'Resource Type: Proced...,[{'Main health record': 'Resource Type: Proced...
3,Q5:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you summarize my care plan history ?,You have two care plans in your record. \n\nTh...,{'Main health record': 'Resource Type: CarePla...,Can you summarize my care plan history ?,Can you summarize my care plan history ?,You have two care plans in your record. \n\nTh...,[{'Main health record': 'Resource Type: CarePl...,[{'Main health record': 'Resource Type: CarePl...,[{'Main health record': 'Resource Type: CarePl...
4,Q1:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,What's my current medications and how should I...,You are currently taking two medications:\n\n1...,{'Main health record': 'Status: Stopped\nInten...,What's my current medications and how should I...,What's my current medications and how should I...,You are currently taking two medications:\n\n1...,"[{'Main health record': 'Status: Stopped, Inte...","[{'Main health record': 'Status: Stopped, Inte...","[{'Main health record': 'Status: Stopped, Inte..."
5,Q4:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,"What are my recent laboratory values, what do ...",I found one record of laboratory values in you...,{'Main health record': 'resourceType: Diagnost...,"What are my recent laboratory values, what do ...","What are my recent laboratory values, what do ...",I found one record of laboratory values in you...,[{'Main health record': 'resourceType: Diagnos...,[{'Main health record': 'resourceType: Diagnos...,[{'Main health record': 'resourceType: Diagnos...
6,Q2:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,"What are my documented allergies, and how seve...",You have the following allergies documented in...,{'Main health record': 'Clinical Status: Activ...,"What are my documented allergies, and how seve...","What are my documented allergies, and how seve...",You have the following allergies documented in...,[{'Main health record': 'Clinical Status: Acti...,[{'Main health record': 'Clinical Status: Acti...,[{'Main health record': 'Clinical Status: Acti...
7,Q3:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you summarize my current medical conditions ?,This is a summary of your medical conditions:\...,{'Main health record': 'Clinical Status: resol...,Can you summarize my current medical condi

In [19]:
# Convert dict to dataset
dataset = Dataset.from_pandas(
    eval_df.loc[eval_df["experiment_id"] == "Q8:gemini-1.5-pro:similarity_search_0_hop"]
)
eval_dataset = EvaluationDataset.from_dict(dataset)

In [16]:
# Convert dict to dataset
dataset = Dataset.from_pandas(eval_df)
eval_dataset = EvaluationDataset.from_dict(dataset)

In [17]:
dataset

Dataset({
    features: ['experiment_id', 'strategy_name', 'full_input', 'output', 'records', 'user_input', 'question', 'response', 'contexts', 'reference_contexts', 'retrieved_contexts'],
    num_rows: 32
})

In [21]:
dataset.to_pandas()

,experiment_id,strategy_name,full_input,output,records,user_input,question,response,contexts,reference_contexts,retrieved_contexts,__index_level_0__
0,Q8:gemini-1.5-pro:similarity_search_0_hop,SimilaritySearch0HopStrategy,Can you summarize my immunization history ?,You received the following immunizations at BL...,A completed immunization was recorded for a pa...,Can you summarize my immunization history ?,Can you summarize my immunization history ?,You received the following immunizations at BL...,[A completed immunization was recorded for a p...,[A completed immunization was recorded for a p...,[A completed immunization was recorded for a p...,0


In [18]:
AZURE_OPENAI_ENDPOINT = ""
AZURE_OPENAI_API_KEY = ""
OPENAI_API_VERSION = "2023-08-01-preview"

In [19]:
PROJECT_ID = "master-experiments-project"
LOCATION_ID = "us-central1"
API_ENDPOINT = "us-central1-aiplatform.googleapis.com"
MODEL_ID = "gemini-1.5-flash-002"

In [20]:
from langchain_core.outputs import ChatGeneration, LLMResult


def gemini_is_finished_parser(response: LLMResult) -> bool:
    is_finished_list = []
    for g in response.flatten():
        resp = g.generations[0][0]

        # Check generation_info first
        if resp.generation_info is not None:
            finish_reason = resp.generation_info.get("finish_reason")
            if finish_reason is not None:
                is_finished_list.append(finish_reason in ["STOP", "MAX_TOKENS"])
                continue

        # Check response_metadata as fallback
        if isinstance(resp, ChatGeneration) and resp.message is not None:
            metadata = resp.message.response_metadata
            if metadata.get("finish_reason"):
                is_finished_list.append(
                    metadata["finish_reason"] in ["STOP", "MAX_TOKENS"]
                )
            elif metadata.get("stop_reason"):
                is_finished_list.append(metadata["stop_reason"] in ["STOP", "MAX_TOKENS"])

        # If no finish reason found, default to True
        if not is_finished_list:
            is_finished_list.append(True)

    return all(is_finished_list)

In [21]:
from langchain_google_vertexai import ChatVertexAI

# llm = AzureChatOpenAI(
#         api_key=AZURE_OPENAI_API_KEY,
#         azure_deployment="gpt-4o-2024-08-06",
#         api_version=OPENAI_API_VERSION,
#         azure_endpoint=AZURE_OPENAI_ENDPOINT,
#     )


# gpt-4o-2024-08-06
# anthropic.claude-v3-sonnet

# llm = config.LLM_MODEL_EVALUATION
llm = ChatVertexAI(model="gemini-1.5-pro", temperature=0, request_parallelism=1)

In [22]:
vertextai_llm = LangchainLLMWrapper(llm, is_finished_parser=gemini_is_finished_parser)

In [23]:
ENDPOINT = "us-central1-aiplatform.googleapis.com"
REGION = "us-central1"
PROJECT_ID = "master-experiments-project"

In [24]:
import vertexai

vertexai.init(project=PROJECT_ID, location=REGION)

In [25]:
from google.auth import default, transport

# Build

credentials, _ = default()
auth_request = transport.requests.Request()
credentials.refresh(auth_request)

In [26]:
from langchain_google_vertexai import VertexAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

# embeddings = AzureOpenAIEmbeddings(
#     model="text-embedding-3-small-1",
#     azure_deployment="text-embedding-3-small-1",
#     api_key=AZURE_OPENAI_API_KEY,
#     api_version=OPENAI_API_VERSION,
#     azure_endpoint=AZURE_OPENAI_ENDPOINT
# )

# embeddings = HuggingFaceBgeEmbeddings(
#             model_name="BAAI/bge-small-en-v1.5"
#         )

ENDPOINT = "us-central1-aiplatform.googleapis.com"
REGION = "us-central1"
PROJECT_ID = "master-experiments-project"

embedding_model = "text-embedding-004"

embedding_function = VertexAIEmbeddings(
    model_name=embedding_model,
    project="master-experiments-project",
    location="us-central1",
)

evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_function)

## Full input

In [27]:
from ragas import RunConfig

# metrics = [LLMContextPrecisionWithoutReference(), ResponseRelevancy(), Faithfulness()]
metrics = [ResponseRelevancy(), Faithfulness()]
# metrics = [Faithfulness()]
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=vertextai_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(timeout=10000, max_workers=50),
)

Evaluating:   0%|          | 0/64 [00:00<?, ?it/s]

Exception raised in Job[34]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[4]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[20]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[28]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[32]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[26]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[41]: AttributeError('StringIO' object has no attribute 'sentences')
Exception raised in Job[52]: AttributeError('StringIO' object has no attribute 'question')
Exception raised in Job[33]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[51]: AttributeError('StringIO' object has no attribute 'sentences')
Exception raised in Job[35]: AttributeError('StringIO' object has no attribute 'stateme

In [29]:
final_data = dataset.to_pandas().merge(
    results.to_pandas(), left_index=True, right_index=True
)

In [30]:
final_data

,experiment_id,strategy_name,full_input,output,records,user_input_x,question,response_x,contexts,reference_contexts_x,retrieved_contexts_x,user_input_y,retrieved_contexts_y,reference_contexts_y,response_y,answer_relevancy,faithfulness
0,Q6:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you provide a breakdown of my medical bills ?,This is a summary of your medical bills:\n\n**...,"{'Main health record': 'Resource Type: Claim,\...",Can you provide a breakdown of my medical bills ?,Can you provide a breakdown of my medical bills ?,This is a summary of your medical bills:\n\n**...,"[{'Main health record': 'Resource Type: Claim,...","[{'Main health record': 'Resource Type: Claim,...","[{'Main health record': 'Resource Type: Claim,...",Can you provide a breakdown of my medical bills ?,"[{'Main health record': 'Resource Type: Claim,...","[{'Main health record': 'Resource Type: Claim,...",This is a summary of your medical bills:\n\n**...,0.775678,0.250000
1,Q8:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you summarize my immunization history ?,Your immunization history shows that you recei...,{'Main health record': 'Resource Type: Immuniz...,Can you summarize my immunization history ?,Can you summarize my immunization history ?,Your immunization history shows that you recei...,[{'Main health record': 'Resource Type: Immuni...,[{'Main health record': 'Resource Type: Immuni...,[{'Main health record': 'Resource Type: Immuni...,Can you summarize my immunization history ?,[{'Main health record': 'Resource Type: Immuni...,[{'Main health record': 'Resource Type: Immuni...,Your immunization history shows that you recei...,0.806250,0.625000
2,Q7:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,"What procedures have I undergone recently, and...","You have undergone several procedures, here is...",{'Main health record': 'Resource Type: Procedu...,"What procedures have I undergone recently, and...","What procedures have I undergone recently, and...","You have undergone several procedures, here is...",[{'Main health record': 'Resource Type: Proced...,[{'Main health record': 'Resource Type: Proced...,[{'Main health record': 'Resource Type: Proced...,"What procedures have I undergone recently, and...",[{'Main health record': 'Resource Type: Proced...,[{'Main health record': 'Resource Type: Proced...,"You have undergone several procedures, here is...",NaN,0.655172
3,Q5:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,Can you summarize my care plan history ?,You have two care plans in your record. \n\nTh...,{'Main health record': 'Resource Type: CarePla...,Can you summarize my care plan history ?,Can you summarize my care plan history ?,You have two care plans in your record. \n\nTh...,[{'Main health record': 'Resource Type: CarePl...,[{'Main health record': 'Resource Type: CarePl...,[{'Main health record': 'Resource Type: CarePl...,Can you summarize my care plan history ?,[{'Main health record': 'Resource Type: CarePl...,[{'Main health record': 'Resource Type: CarePl...,You have two care plans in your record. \n\nTh...,0.732787,NaN
4,Q1:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,What's my current medications and how should I...,You are currently taking two medications:\n\n1...,{'Main health record': 'Status: Stopped\nInten...,What's my current medications and how should I...,What's my current medications and how should I...,You are currently taking two medications:\n\n1...,"[{'Main health record': 'Status: Stopped, Inte...","[{'Main health record': 'Status: Stopped, Inte...","[{'Main health record': 'Status: Stopped, Inte...",What's my current medications and how should I...,"[{'Main health record': 'Status: Stopped, Inte...","[{'Main health record': 'Status: Stopped, Inte...",You are currently taking two medications:\n\n1...,0.674474,NaN
5,Q4:gemini-1.5-pro:lexical_search_1_hop,LexicalSearch1HopStrategy,"What are my recent laboratory values, what do ...",I found on

In [31]:
!pwd

/home/baptvit/Documents/mestrado/master-experiments/evaluations


I0000 00:00:1734215098.379274   22499 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


In [32]:
final_data.to_csv(
    "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/gold/Beatris270_Bogan287_ragas.csv"
)

In [ ]:
from ragas import RunConfig

# metrics = [LLMContextPrecisionWithoutReference(), ResponseRelevancy(), Faithfulness()]
metrics = [Faithfulness()]
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=vertextai_llm,
    embeddings=embeddings,
    run_config=RunConfig(timeout=10000, max_workers=1),
    raise_exceptions=True,
)

In [25]:
dataset.to_pandas().merge(results.to_pandas(), left_index=True, right_index=True)

,experiment_id,strategy_name,full_input,output,records,user_input_x,question,response_x,contexts,reference_contexts_x,retrieved_contexts_x,__index_level_0__,user_input_y,retrieved_contexts_y,reference_contexts_y,response_y,faithfulness
0,Q8:gemini-1.5-pro:similarity_search_0_hop,SimilaritySearch0HopStrategy,Can you summarize my immunization history ?,You received the following immunizations at BL...,A completed immunization was recorded for a pa...,Can you summarize my immunization history ?,Can you summarize my immunization history ?,You received the following immunizations at BL...,[A completed immunization was recorded for a p...,[A completed immunization was recorded for a p...,[A completed immunization was recorded for a p...,0,Can you summarize my immunization history ?,[A completed immunization was recorded for a p...,[A completed immunization was recorded for a p...,You received the following immunizations at BL...,0.666667


## Llama 

In [ ]:
from ragas import RunConfig

# metrics = [LLMContextPrecisionWithoutReference(), ResponseRelevancy(), Faithfulness()]
metrics = [Faithfulness()]
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=llama_llm,
    embeddings=embeddings,
    run_config=RunConfig(timeout=10000, max_workers=1),
    raise_exceptions=True,
)

In [ ]:
dataset.to_pandas().merge(results.to_pandas(), left_index=True, right_index=True)